# Now that we've established the best workflow, we have to construct the datasets and surveys.

# To do so, we must collect images surveys for all items via a Google Search API Key

In [ ]:
# Now that we've established the best workflow, we have to construct the datasets and surveys.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [1]:
from google.colab import userdata
my_search_key = userdata.get('GOOGLE_SEARCH_KEY')

if my_search_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time

!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.3 MB/s eta 0:00:00
time: 298 µs (started: 2026-01-06 19:09:27 +00:00)


In [12]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"
!cd LLM4BEAR && git sparse-checkout add "3_Human Evaluation"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 26 (delta 1), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 19.77 KiB | 449.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 82 (delta 7), reused 81 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 46.13 MiB | 21.04 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (83/83), done.
remote: Enumerating objects: 436, done.
remote: Counting objects: 100% (436/436), done.
remote: Compressing obj

In [3]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

time: 110 ms (started: 2026-01-06 19:09:37 +00:00)


In [ ]:
with open("/content/LLM4BEAR/BundleRec Data/enriched_outputs_electronic.pkl", "rb") as f:
    text_electronics = pickle.load(f)

print(text_electronics[3498])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_clothing.pkl", "rb") as f:
    text_clothing = pickle.load(f)

print(text_clothing[3000])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_food.pkl", "rb") as f:
    text_food = pickle.load(f)

print(text_food[3000])


electronic_items = [electronic_metadata.iloc[i]['titles'] for i in range(len(electronic_metadata))]

food_items = [food_metadata.iloc[i]['titles'] for i in range(len(food_metadata))]

clothing_items = [clothing_metadata.iloc[i]['titles'] for i in range(len(clothing_metadata))]

In [ ]:
import os
for folder in ["electronic", "food", "clothing"]:
    os.makedirs(f"/content/drive/MyDrive/bundles/images/{folder}", exist_ok=True)


In [ ]:
# Colab: run this cell
import os, csv, time, io, requests, random
from PIL import Image
from google.colab import userdata

API_guy = userdata.get('google_shit_key')         # you already set this
CX = "0453b6ec5452e41e1"                             # <-- put your CSE ID here

SAVE_ROOT = "/content/drive/MyDrive/bundles/images"


CANDIDATES_PER_QUERY = 5
MIN_W, MIN_H = 256, 256
SLEEP_BETWEEN_CALLS = 0.25  # tune this if you hit quota/429s

def google_image_search(q, num=CANDIDATES_PER_QUERY, start=1):
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "q": q, "cx": CX, "key": API_guy,
        "searchType": "image", "num": min(10, num), "start": start,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("items", []) or []

def fetch_bytes(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.content, r.headers.get("Content-Type","").split(";")[0].lower()

def save_as_jpg(content_bytes, outpath):
    # convert to RGB JPG regardless of original type (png/webp/etc.)
    with Image.open(io.BytesIO(content_bytes)) as im:
        if im.mode not in ("RGB","L"):
            im = im.convert("RGB")
        elif im.mode == "L":
            im = im.convert("RGB")
        # reject tiny images
        w, h = im.size
        if w < MIN_W or h < MIN_H:
            return False
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        im.save(outpath, format="JPEG", quality=90, optimize=True)
        return True

def process_category(cat, items, prefix):
    cat_dir = os.path.join(SAVE_ROOT, cat)
    os.makedirs(cat_dir, exist_ok=True)
    manifest = os.path.join(SAVE_ROOT, f"{cat}_manifest.csv")
    write_header = not os.path.exists(manifest)

    with open(manifest, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["index_1_based", "title", "filename", "source_url", "status"])

        for i, title in items:
            stem = f"{prefix}_{i + 1}"  # global 1-based index
            outjpg = os.path.join(cat_dir, f"{stem}.jpg")
            status, src = "missing", ""

            # skip if already exists (idempotent re-runs)
            if os.path.exists(outjpg):
                w.writerow([i + 1, title, os.path.basename(outjpg), "", "already_exists"])
                continue

            try:
                cands = google_image_search(title, num=CANDIDATES_PER_QUERY, start=1)
                tried_pages = 0
                while tried_pages < 2 and not os.path.exists(outjpg):
                    for it in cands:
                        src = it.get("link", "")
                        try:
                            bytes_, ct = fetch_bytes(src)
                            if save_as_jpg(bytes_, outjpg):
                                status = "ok"
                                break
                        except Exception:
                            continue
                    tried_pages += 1
                    if status != "ok":
                        # try next page (11..20)
                        cands = google_image_search(title, num=CANDIDATES_PER_QUERY, start=11)

                if status != "ok":
                    status = "no_suitable_image"
                    if os.path.exists(outjpg):
                        os.remove(outjpg)

            except requests.HTTPError as e:
                status = f"http_error_{getattr(e.response, 'status_code', 'NA')}"
            except Exception as e:
                status = f"error:{type(e).__name__}"

            w.writerow([i + 1, title, os.path.basename(outjpg) if status == "ok" else "", src, status])
            time.sleep(SLEEP_BETWEEN_CALLS + random.random() * 0.5)

    print(f"[{cat}] Done. Manifest: {manifest}")


In [ ]:
batch_size = 100

# --- ELECTRONIC ---
n = len(electronic_metadata)
batches = [(i, min(i + batch_size, n)) for i in range(0, n, batch_size)]
print(f"Total batches: {len(batches)}")
print(batches[-1])  # preview first few


# 🚀 Run all batches for electronics only
for (start, end) in batches:
    print(f"\n⚙️ Processing electronics batch {start}-{end}...")
    sample = [(i, electronic_items[i]) for i in range(start, end)]
    process_category("electronic", sample, "electronic")
    print(f"✅ Batch {start}-{end} done.\n")

print("🎉 All electronics batches completed.")

In [ ]:
batch_size = 100

# --- CLOTHING ---
n_clothing = len(clothing_metadata)
batches_clothing = [(i, min(i + batch_size, n_clothing)) for i in range(0, n_clothing, batch_size)]
print(f"👕 Clothing: {len(batches_clothing)} batches total.")

for (start, end) in batches_clothing:
    print(f"\n⚙️ Processing CLOTHING batch {start}-{end}...")
    sample = [(i, clothing_items[i]) for i in range(start, end)]
    process_category("clothing", sample, "clothing")
    print(f"✅ Done CLOTHING batch {start}-{end}.\n")



In [ ]:
batch_size = 100

# --- FOOD ---
n_food = len(food_metadata)
batches_food = [(i, min(i + batch_size, n_food)) for i in range(0, n_food, batch_size)]
print(f"🍎 Food: {len(batches_food)} batches total.")

for (start, end) in batches_food:
    print(f"\n⚙️ Processing FOOD batch {start}-{end}...")
    sample = [(i, food_items[i]) for i in range(start, end)]
    process_category("food", sample, "food")
    print(f"✅ Done FOOD batch {start}-{end}.\n")


In [ ]:
# === Check for missing images across categories ===
import os, re
import pandas as pd

SAVE_ROOT = "/content/drive/MyDrive/bundles/images"

def summarize_missing(category, items):
    cat_dir = os.path.join(SAVE_ROOT, category)
    manifest_path = os.path.join(SAVE_ROOT, f"{category}_manifest.csv")

    # 1) expected indices from metadata length (1-based)
    total_expected = len(items)
    expected = set(range(1, total_expected + 1))

    # 2) what exists on disk
    on_disk = set()
    if os.path.isdir(cat_dir):
        for f in os.listdir(cat_dir):
            m = re.match(fr"{category}_(\d+)\.jpg$", f)
            if m:
                on_disk.add(int(m.group(1)))

    # 3) manifest "ok" rows (if manifest exists)
    ok_in_manifest = set()
    non_ok_rows = []
    if os.path.exists(manifest_path):
        try:
            df = pd.read_csv(manifest_path)
            if {"index_1_based","status"}.issubset(df.columns):
                ok_in_manifest = set(df.loc[df["status"].eq("ok"), "index_1_based"].astype(int).tolist())
                non_ok_rows = df.loc[df["status"].ne("ok"), ["index_1_based","title","status"]]
        except Exception:
            pass

    # 4) missing = expected but not present as a good file
    #     (require the file to exist; manifest is informative but not required)
    missing_by_file = sorted(list(expected - on_disk))

    # Also useful: “failed/needs retry” = rows not ok in manifest
    failed_from_manifest = sorted(list(set(non_ok_rows["index_1_based"].astype(int).tolist()))) if len(non_ok_rows) else []

    print(f"\n[{category}]")
    print(f"  expected: {total_expected}")
    print(f"  on_disk:  {len(on_disk)} (max={max(on_disk) if on_disk else 0})")
    print(f"  missing (by file): {len(missing_by_file)}")
    if missing_by_file[:10]:
        print(f"  sample missing: {missing_by_file[:10]}{' ...' if len(missing_by_file) > 10 else ''}")
    if failed_from_manifest:
        print(f"  non-ok in manifest: {len(failed_from_manifest)} (first 10: {failed_from_manifest[:10]})")

    # Optional: save CSV of missing indices to Drive for this category
    out_csv = os.path.join(SAVE_ROOT, f"{category}_missing_indices.csv")
    pd.DataFrame({"index_1_based": missing_by_file}).to_csv(out_csv, index=False)
    print(f"  👉 wrote missing list to: {out_csv}")

    return missing_by_file

# ---- Run checks (pass your loaded item lists) ----
missing_e = summarize_missing("electronic", electronic_items)
missing_c = summarize_missing("clothing",   clothing_items)
missing_f = summarize_missing("food",       food_items)


# Double check that we've found a corresponding image for all items.

In [ ]:
for i in missing_e:
    print(i, ":", electronic_items[i-1])

for i in missing_c:
    print(i, ":", clothing_items[i-1])

for i in missing_f:
    print(i, ":", food_items[i-1])

# Put all images onto a public Google Drive folder, such that anyone can access the item images if given a link.

In [ ]:
# --- SETUP ---
!pip install --quiet google-api-python-client

import json
from google.colab import auth
auth.authenticate_user()  # authorizes your account for Drive access

from googleapiclient.discovery import build

# Folder IDs (from your links)
folder_ids = {
    "electronics": "1ES_vPiziSWfL5RLiMQ8F6DjUnu7iUcPy",
    "clothing": "1gt0L8TboCm33HhAlzBxjenC04mPi4KGb",
    "food": "1p4XeMmfv0ZkyY7p00VaKC072FtJB-tOB"
}

# Build Drive client
drive = build("drive", "v3")

def make_public(folder_id):
    try:
        drive.permissions().create(
            fileId=folder_id,
            body={"role": "reader", "type": "anyone", "allowFileDiscovery": False},
        ).execute()
        print(f"✓ Folder {folder_id} set to 'anyone with the link'")
    except Exception as e:
        print(f"! Could not update {folder_id}: {e}")

# --- MAKE FOLDERS PUBLIC ---
for f in folder_ids.values():
    make_public(f)

# --- SAVE CONFIG JSON (optional reference) ---
with open("folders_public.json", "w") as f:
    json.dump(folder_ids, f, indent=2)

print("\n✅ Done — all folders should now be public-at-link.")
print("Any image inside these folders is reachable at:")
print("https://drive.google.com/uc?export=view&id=<FILE_ID>")


In [ ]:
# === SETUP ===
!pip install --quiet google-api-python-client

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
import pandas as pd

# Your folder IDs
FOLDERS = {
    "electronics": "1ES_vPiziSWfL5RLiMQ8F6DjUnu7iUcPy",
    "clothing": "1gt0L8TboCm33HhAlzBxjenC04mPi4KGb",
    "food": "1p4XeMmfv0ZkyY7p00VaKC072FtJB-tOB"
}

drive = build("drive", "v3")

def list_all_files(folder_id):
    """Return list of {name, id} for all image files in a folder."""
    files = []
    page_token = None
    while True:
        response = drive.files().list(
            q=f"'{folder_id}' in parents and mimeType contains 'image/' and trashed=false",
            fields="nextPageToken, files(id, name)",
            pageSize=1000,
            pageToken=page_token
        ).execute()
        files.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return files

# === RETRIEVE ALL FILES ===
all_data = {}

for label, fid in FOLDERS.items():
    print(f"Listing {label}...")
    files = list_all_files(fid)
    print(f"  → {len(files)} images found.")
    df = pd.DataFrame(files)
    df["imageLink"] = "https://drive.google.com/uc?export=view&id=" + df["id"]
    df.to_csv(f"{label}_files.csv", index=False)
    all_data[label] = df

# Combine into one big table
combined = pd.concat(all_data, names=["domain", "index"]).reset_index(level=0)
combined.to_csv("all_folders_files.csv", index=False)
print("\n✅ Done! Exported:")
print(" - electronics_files.csv")
print(" - clothing_files.csv")
print(" - food_files.csv")
print(" - all_folders_files.csv")
print("\nEach file has columns: name, id, imageLink")

combined.head(10)


In [ ]:
# After you've created df (the pandas DataFrame with columns name, id, imageLink)
import pandas as pd
import re

def extract_number(filename):
    m = re.search(r"(\d+)", filename)
    return int(m.group(1)) if m else -1

df["num"] = df["name"].apply(extract_number)
df = df.sort_values("num").drop(columns="num").reset_index(drop=True)

# Re-save the file in correct order
df.to_csv("electronics_files_sorted.csv", index=False)
df.head(10)


In [ ]:
for label in ["electronics", "clothing", "food"]:
    df = pd.read_csv(f"{label}_files.csv")
    df["num"] = df["name"].str.extract(r"(\d+)").astype(int)
    df.sort_values("num", inplace=True)
    df.drop(columns="num", inplace=True)
    df.to_csv(f"{label}_files_sorted.csv", index=False)
    print(f"✓ {label} sorted and saved")


# Best workflow used to make surveys

In [5]:
def flag_guys(flags):

    l = len(flags[0])

    first_flags = flags[0]
    last_flags = flags[-1]

    regular_flags = [True for _ in range(l)]
    for i in range(l):
        if first_flags[i] == False:
            regular_flags[i] = False
        if last_flags[i] == True:
            regular_flags[i] = False


    empty = []

    for i in range(l):
        if regular_flags[i] == True:
            empty.append(i)


    return empty




time: 1.37 ms (started: 2026-01-06 19:09:38 +00:00)


In [9]:
domains = ["electronic", "clothing", "food"]



# with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_electronic_complete_electronic_no_graph_help_run.pkl", "rb") as f:
# with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/electronic_bundles_refined.pkl", "rb") as f:
with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/historical_bundle_changes_electronic_complete_electronic_no_graph_help_run.pkl", "rb") as f:

    electronic_intents, electronic_bundle_items, electronic_bundle_indices, electronic_scores, electronic_min_scores, electronic_flags = pickle.load(f)

electronic_mod_indices = flag_guys(electronic_flags)

# with open(f"/content/drive/My Drive/Bundle_Refinement/historical_bundle_changes_clothing_complete_clothing_no_graph_help_run.pkl", "rb") as f:
# with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/clothing_bundles_refined.pkl", "rb") as f:
with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/historical_bundle_changes_clothing_complete_clothing_no_graph_help_run.pkl", "rb") as f:

    clothing_intents, clothing_bundle_items, clothing_bundle_indices, clothing_scores, clothing_min_scores, clothing_flags = pickle.load(f)


clothing_mod_indices = flag_guys(clothing_flags)

# with open(f"/content/drive/My Drive/Bundle_Refinement/historical_bundle_changes_food_complete_food_no_graph_help_run.pkl", "rb") as f:
# with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/food_bundles_refined.pkl", "rb") as f:
with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/historical_bundle_changes_food_complete_food_no_graph_help_run.pkl", "rb") as f:
    food_intents, food_bundle_items, food_bundle_indices, food_scores, food_min_scores, food_flags = pickle.load(f)

food_mod_indices = flag_guys(food_flags)


time: 174 ms (started: 2026-01-06 19:17:02 +00:00)


In [10]:
cleaned_electronic_mod_indices = []
cleaned_clothing_mod_indices = []
cleaned_food_mod_indices = []


for i in electronic_mod_indices:
    if set(electronic_bundle_indices[0][i]) != set(electronic_bundle_indices[-1][i]):
        cleaned_electronic_mod_indices.append(i)

for i in clothing_mod_indices:
    if set(clothing_bundle_indices[0][i]) != set(clothing_bundle_indices[-1][i]):
        cleaned_clothing_mod_indices.append(i)

for i in food_mod_indices:
    if set(food_bundle_indices[0][i]) != set(food_bundle_indices[-1][i]):
        cleaned_food_mod_indices.append(i)

time: 5.24 ms (started: 2026-01-06 19:18:18 +00:00)


In [11]:
initial_electronic_bundle_indices = electronic_bundle_indices[0]
initial_clothing_bundle_indices = clothing_bundle_indices[0]
initial_food_bundle_indices = food_bundle_indices[0]

final_electronic_bundle_indices = electronic_bundle_indices[-1]
final_clothing_bundle_indices = clothing_bundle_indices[-1]
final_food_bundle_indices = food_bundle_indices[-1]


survey_initial_electronic_bundle_indices = [initial_electronic_bundle_indices[i] for i in cleaned_electronic_mod_indices]
survey_initial_clothing_bundle_indices = [initial_clothing_bundle_indices[i] for i in cleaned_clothing_mod_indices]
survey_initial_food_bundle_indices = [initial_food_bundle_indices[i] for i in cleaned_food_mod_indices]

survey_final_electronic_bundle_indices = [final_electronic_bundle_indices[i] for i in cleaned_electronic_mod_indices]
survey_final_clothing_bundle_indices = [final_clothing_bundle_indices[i] for i in cleaned_clothing_mod_indices]
survey_final_food_bundle_indices = [final_food_bundle_indices[i] for i in cleaned_food_mod_indices]



time: 3.47 ms (started: 2026-01-06 19:18:27 +00:00)


In [ ]:
import json, re, pandas as pd

for i in range(1, 48):

    charizard = f"clothing_batch_{i}"

    IN_PATH = f"/content/drive/My Drive/bundles/surveys/{charizard}.json"
    OUT_CSV = f"/content/drive/My Drive/bundles/randomise_mapping/{charizard}_randomise_map.csv"

    # Regex to capture the hidden marker we injected, e.g.
    # <!-- randomise: yes; left=final; right=initial -->
    MARKER_RE = re.compile(r"<!--\s*randomise:\s*([^;>\s]+)\s*;?\s*(?:left=([^;>\s]+))?\s*;?\s*(?:right=([^;>\s]+))?\s*-->",
                          re.IGNORECASE)

    with open(IN_PATH, "r") as f:
        survey = json.load(f)

    rows = []
    for page in survey.get("pages", []):
        page_name = page.get("name", "")
        pair_idx = None
        # Try to pull the pair index if it's in the name like "food_pair_1201"
        m = re.search(r"_pair_(\d+)", page_name)
        if m:
            pair_idx = int(m.group(1))

        # Find the HTML element and extract the marker
        html_elems = [e for e in page.get("elements", []) if e.get("type") == "html"]
        marker = None
        left_map = right_map = None
        if html_elems:
            html = html_elems[0].get("html", "")
            mm = MARKER_RE.search(html)
            if mm:
                marker = mm.group(1).lower()              # "yes" / "no"
                left_map  = (mm.group(2) or "").lower()   # "initial" / "final" (if present)
                right_map = (mm.group(3) or "").lower()

        rows.append({
            "page_name": page_name,
            "pair_index": pair_idx,
            "randomised": marker,          # "yes" or "no"
            "left_is": left_map,           # initial/final (when present)
            "right_is": right_map          # initial/final (when present)
        })

    df = pd.DataFrame(rows)
    display(df.head(20))

    # Summary counts
    print("\nCounts:")
    print(df["randomised"].value_counts(dropna=False))

    # Save CSV
    df.to_csv(OUT_CSV, index=False)
    print("\nSaved:", OUT_CSV)


In [ ]:
import json, re, pandas as pd

for i in range(1, 63):

    charizard = f"electronics_batch_{i}"

    IN_PATH = f"/content/drive/My Drive/bundles/surveys/{charizard}.json"
    OUT_CSV = f"/content/drive/My Drive/bundles/randomise_mapping/{charizard}_randomise_map.csv"

    # Regex to capture the hidden marker we injected, e.g.
    # <!-- randomise: yes; left=final; right=initial -->
    MARKER_RE = re.compile(r"<!--\s*randomise:\s*([^;>\s]+)\s*;?\s*(?:left=([^;>\s]+))?\s*;?\s*(?:right=([^;>\s]+))?\s*-->",
                          re.IGNORECASE)

    with open(IN_PATH, "r") as f:
        survey = json.load(f)

    rows = []
    for page in survey.get("pages", []):
        page_name = page.get("name", "")
        pair_idx = None
        # Try to pull the pair index if it's in the name like "food_pair_1201"
        m = re.search(r"_pair_(\d+)", page_name)
        if m:
            pair_idx = int(m.group(1))

        # Find the HTML element and extract the marker
        html_elems = [e for e in page.get("elements", []) if e.get("type") == "html"]
        marker = None
        left_map = right_map = None
        if html_elems:
            html = html_elems[0].get("html", "")
            mm = MARKER_RE.search(html)
            if mm:
                marker = mm.group(1).lower()              # "yes" / "no"
                left_map  = (mm.group(2) or "").lower()   # "initial" / "final" (if present)
                right_map = (mm.group(3) or "").lower()

        rows.append({
            "page_name": page_name,
            "pair_index": pair_idx,
            "randomised": marker,          # "yes" or "no"
            "left_is": left_map,           # initial/final (when present)
            "right_is": right_map          # initial/final (when present)
        })

    df = pd.DataFrame(rows)
    display(df.head(20))

    # Summary counts
    print("\nCounts:")
    print(df["randomised"].value_counts(dropna=False))

    # Save CSV
    df.to_csv(OUT_CSV, index=False)
    print("\nSaved:", OUT_CSV)


In [ ]:
import json, re, pandas as pd

for i in range(1, 63):

    charizard = f"food_batch_{i}"

    IN_PATH = f"/content/drive/My Drive/bundles/surveys/{charizard}.json"
    OUT_CSV = f"/content/drive/My Drive/bundles/randomise_mapping/{charizard}_randomise_map.csv"

    # Regex to capture the hidden marker we injected, e.g.
    # <!-- randomise: yes; left=final; right=initial -->
    MARKER_RE = re.compile(r"<!--\s*randomise:\s*([^;>\s]+)\s*;?\s*(?:left=([^;>\s]+))?\s*;?\s*(?:right=([^;>\s]+))?\s*-->",
                          re.IGNORECASE)

    with open(IN_PATH, "r") as f:
        survey = json.load(f)

    rows = []
    for page in survey.get("pages", []):
        page_name = page.get("name", "")
        pair_idx = None
        # Try to pull the pair index if it's in the name like "food_pair_1201"
        m = re.search(r"_pair_(\d+)", page_name)
        if m:
            pair_idx = int(m.group(1))

        # Find the HTML element and extract the marker
        html_elems = [e for e in page.get("elements", []) if e.get("type") == "html"]
        marker = None
        left_map = right_map = None
        if html_elems:
            html = html_elems[0].get("html", "")
            mm = MARKER_RE.search(html)
            if mm:
                marker = mm.group(1).lower()              # "yes" / "no"
                left_map  = (mm.group(2) or "").lower()   # "initial" / "final" (if present)
                right_map = (mm.group(3) or "").lower()

        rows.append({
            "page_name": page_name,
            "pair_index": pair_idx,
            "randomised": marker,          # "yes" or "no"
            "left_is": left_map,           # initial/final (when present)
            "right_is": right_map          # initial/final (when present)
        })

    df = pd.DataFrame(rows)
    display(df.head(20))

    # Summary counts
    print("\nCounts:")
    print(df["randomised"].value_counts(dropna=False))

    # Save CSV
    df.to_csv(OUT_CSV, index=False)
    print("\nSaved:", OUT_CSV)


In [ ]:
CSV_PATHS = {
    "electronics": "/content/LLM4BEAR/3_Human Evaluation/electronic_files_sorted.csv",
    "clothing":    "/content/LLM4BEAR/3_Human Evaluation/clothing_files_sorted.csv",
    "food":        "/content/LLM4BEAR/3_Human Evaluation/food_files_sorted.csv",
}

In [ ]:
DOMAIN   = "electronics"   # one of: "electronics", "clothing", "food"
PAIR_IDX = 0

import re, json, pandas as pd

# ---- load CSV for the domain and build number->URL map (number is 1-based from filename) ----
df = pd.read_csv(CSV_PATHS[DOMAIN])
# expect columns: name, id, imageLink (from your earlier export)
def num_from_name(name: str) -> int:
    m = re.search(r"(\d+)", str(name))
    return int(m.group(1)) if m else None

num2url = { num_from_name(r["name"]): r["imageLink"] for _, r in df.iterrows() }

# ---- helper: convert a list of ZERO-BASED indices -> image URLs (by adding +1) ----
def indices_to_urls_zero_based(idx_list):
    urls = []
    for i in idx_list:
        n = i + 1  # +1 because filenames are 1-based (electronic_1.jpg, etc.)
        try:
            urls.append(num2url[n])
        except KeyError:
            raise KeyError(f"No URL for filename number {n} in {DOMAIN} (check CSV).")
    return urls

# ---- pick the pair k = PAIR_IDX from your in-memory lists ----
if DOMAIN == "electronics":
    init_idx_vec = survey_initial_electronic_bundle_indices[PAIR_IDX]
    final_idx_vec = survey_final_electronic_bundle_indices[PAIR_IDX]
elif DOMAIN == "clothing":
    init_idx_vec = survey_initial_clothing_bundle_indices[PAIR_IDX]
    final_idx_vec = survey_final_clothing_bundle_indices[PAIR_IDX]
elif DOMAIN == "food":
    init_idx_vec = survey_initial_food_bundle_indices[PAIR_IDX]
    final_idx_vec = survey_final_food_bundle_indices[PAIR_IDX]
else:
    raise ValueError("DOMAIN must be one of electronics/clothing/food")

# ---- map to URLs (this is the only thing you said you need) ----
init_urls  = indices_to_urls_zero_based(init_idx_vec)
final_urls = indices_to_urls_zero_based(final_idx_vec)

# ---- emit a tiny JSON for THIS comparison pair (handy to feed SurveyJS later) ----
pair_json = {
    "domain": DOMAIN,
    "pair_index": PAIR_IDX,
    "initial": {
        "indices_zero_based": init_idx_vec,
        "indices_filename_numbers": [i+1 for i in init_idx_vec],
        "urls": init_urls
    },
    "final": {
        "indices_zero_based": final_idx_vec,
        "indices_filename_numbers": [i+1 for i in final_idx_vec],
        "urls": final_urls
    }
}

out_path = f"{DOMAIN}_pair_{PAIR_IDX}.json"
with open(out_path, "w") as f:
    json.dump(pair_json, f, indent=2)

print(f"✓ wrote {out_path}")
print("\nInitial URLs:")

for u in init_urls:
    print(" ", u)
print("\nFinal URLs:")
for u in final_urls:
    print(" ", u)




print()
print()

for i in survey_initial_electronic_bundle_indices[PAIR_IDX]:
    print(electronic_metadata.iloc[i]['titles'])
    print(text_electronics[i])
print()
print()

for i in survey_final_electronic_bundle_indices[PAIR_IDX]:
    print(electronic_metadata.iloc[i]['titles'])
    print(text_electronics[i])

In [ ]:
# --- Config: your CSVs ---
CSV_PATHS = {
    "electronics": "/content/LLM4BEAR/3_Human Evaluation/electronic_files_sorted.csv",
    "clothing":    "/content/LLM4BEAR/3_Human Evaluation/clothing_files_sorted.csv",
    "food":        "/content/LLM4BEAR/3_Human Evaluation/food_files_sorted.csv",
}

# --- Code ---
import re
import pandas as pd
from urllib.parse import urlparse, parse_qs

ID_RE = re.compile(r'^[-\w]{10,}$')  # conservative Drive ID check

def extract_drive_id(u: str) -> str | None:
    if not isinstance(u, str) or not u:
        return None
    try:
        p = urlparse(u)
    except Exception:
        return None
    if p.netloc != "drive.google.com":
        return None

    path = p.path or ""
    qs = parse_qs(p.query or "")

    # Already a thumbnail? accept as-is
    if path == "/thumbnail" and "id" in qs and ID_RE.match(qs["id"][0] or ""):
        return qs["id"][0]

    # /uc?export=view|download&id=FILE_ID
    if path == "/uc":
        exp = (qs.get("export", [""])[0] or "").lower()
        fid = qs.get("id", [""])[0]
        if exp in ("view", "download") and ID_RE.match(fid):
            return fid

    # /open?id=FILE_ID
    if path == "/open":
        fid = qs.get("id", [""])[0]
        if ID_RE.match(fid):
            return fid

    # /file/d/FILE_ID/(view|preview|edit)...
    m = re.match(r"^/file/d/([^/]+)(?:/|$)", path)
    if m and ID_RE.match(m.group(1)):
        return m.group(1)

    return None

def to_thumbnail(u: str, width: int = 320) -> str:
    fid = extract_drive_id(u)
    if not fid:
        # leave untouched if not a recognized Drive URL (1-to-1 safe)
        return u
    return f"https://drive.google.com/thumbnail?id={fid}&sz=w{width}"

def process_csv(path: str, overwrite_col_c: bool = True, width: int = 320) -> str:
    # Read as strings to avoid losing URL format
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if df.shape[1] < 3:
        raise ValueError(f"{path}: expected at least 3 columns (column C is the 3rd)")

    col_c = df.columns[2]  # 0-based index -> third column
    original = df[col_c].copy()

    # Transform column C safely
    df[col_c] = df[col_c].map(lambda s: to_thumbnail(s, width=width))

    # Optional: sanity check — only change when we truly recognized an ID
    changed_mask = df[col_c] != original
    recognized = sum(changed_mask)
    print(f"{path}: converted {recognized} of {len(df)} rows in column C")

    # Save alongside the original
    out_path = path.replace(".csv", "_thumbs.csv")
    df.to_csv(out_path, index=False)
    return out_path

# --- Run for all files ---
outputs = []
for name, path in CSV_PATHS.items():
    try:
        out = process_csv(path, overwrite_col_c=True, width=320)
        outputs.append((name, out))
    except Exception as e:
        print(f"ERROR processing {name}: {e}")

print("Saved files:")
for name, out in outputs:
    print(f"- {name}: {out}")


In [ ]:
import re, json, pandas as pd

# === CSV paths ===
CSV_PATHS = {
    "electronics": "/content/LLM4BEAR/3_Human Evaluation/electronic_files_sorted_thumbs.csv",
    "clothing":    "/content/LLM4BEAR/3_Human Evaluation/clothing_files_sorted_thumbs.csv",
    "food":        "/content/LLM4BEAR/3_Human Evaluation/food_files_sorted_thumbs.csv",
}

def num_from_name(name):
    m = re.search(r"(\d+)", str(name))
    return int(m.group(1)) if m else None

def build_num2url(domain):
    df = pd.read_csv(CSV_PATHS[domain])
    return {num_from_name(r["name"]): r["imageLink"] for _, r in df.iterrows()}

def indices_to_urls_zero_based(domain, idx_list, num2url):
    return [num2url[i + 1] for i in idx_list]  # +1 offset

# === load lookups ===
maps = {d: build_num2url(d) for d in ["electronics", "clothing", "food"]}

def make_item_block(url, title, desc):
    return f"""
    <div style='margin-bottom:12px;'>
      <img src='{url}' style='max-width:100%;border-radius:8px;margin-bottom:6px;'>
      <b>{title}</b><br>
      <small>{desc}</small>
    </div>
    """


def make_bundle_html(domain, pair_idx, init_idx_vec, final_idx_vec, titles, texts):
    init_urls  = indices_to_urls_zero_based(domain, init_idx_vec, maps[domain])
    final_urls = indices_to_urls_zero_based(domain, final_idx_vec, maps[domain])

    init_html = "".join(
        make_item_block(init_urls[j], titles[i], texts[i])
        for j, i in enumerate(init_idx_vec)
    )
    final_html = "".join(
        make_item_block(final_urls[j], titles[i], texts[i])
        for j, i in enumerate(final_idx_vec)
    )

    # ✅ Moderated width (fits SurveyJS better)
    return f"""
    <div style='
        display:flex;
        gap:32px;
        justify-content:center;
        align-items:flex-start;
        flex-wrap:nowrap;
        max-width:1100px;
        margin:0 auto;
    '>
      <div style='flex:1;min-width:450px;'>
        <h4 style="text-align:center;">Bundle 1</h4>
        {init_html}
      </div>
      <div style='flex:1;min-width:450px;'>
        <h4 style="text-align:center;">Bundle 2</h4>
        {final_html}
      </div>
    </div>
    """


def build_pages_for_domain(domain, survey_init, survey_final, titles, texts):
    """
    Clean version for SurveyJS:
    - Randomises left/right 50% of the time
    - Stores randomise flag as HTML comment (invisible to user)
    - Uses 'rateStep' instead of 'step'
    - No invalid SurveyJS keys
    """
    pages = []

    for k in range(len(survey_init)):
        # flip 50% of the time
        randomised = random.random() < 0.5

        if randomised:
            left_idx_vec  = survey_final[k]
            right_idx_vec = survey_init[k]
            left_label, right_label = "Bundle 2", "Bundle 1"
            randomise_flag = "yes"
        else:
            left_idx_vec  = survey_init[k]
            right_idx_vec = survey_final[k]
            left_label, right_label = "Bundle 1", "Bundle 2"
            randomise_flag = "no"

        # build HTML with invisible randomise marker
        left_html  = make_bundle_html(domain, k, left_idx_vec,  [], titles, texts)
        right_html = make_bundle_html(domain, k, right_idx_vec, [], titles, texts)
        html = f"""
        <!-- randomise: {randomise_flag} -->
        <div style='display:flex;gap:24px;flex-wrap:wrap;align-items:flex-start;'>
          <div style='flex:1;min-width:280px;'>
            <h4>{left_label}</h4>
            {left_html}
          </div>
          <div style='flex:1;min-width:280px;'>
            <h4>{right_label}</h4>
            {right_html}
          </div>
        </div>
        """

        # assemble the SurveyJS page
        pages.append({
            "name": f"{domain}_pair_{k}",
            "elements": [
                {
                    "type": "html",
                    "name": f"{domain}_pair_{k}_html",
                    "html": html
                },
                {
                    "type": "radiogroup",
                    "name": f"{domain}_pair_{k}_choice",
                    "title": "Which bundle is better — bundle 1 or bundle 2?",
                    "choices": [
                        {"value": "bundle_1", "text": "Bundle 1 is better"},
                        {"value": "bundle_2", "text": "Bundle 2 is better"}
                    ],
                    "isRequired": True
                },
                {
                    "type": "rating",
                    "name": f"{domain}_pair_{k}_bundle1_rating",
                    "title": "Rate bundle 1 out of 5",
                    "rateMin": 1,
                    "rateMax": 5,
                    "rateStep": 1,
                    "isRequired": True
                },
                {
                    "type": "rating",
                    "name": f"{domain}_pair_{k}_bundle2_rating",
                    "title": "Rate bundle 2 out of 5",
                    "rateMin": 1,
                    "rateMax": 5,
                    "rateStep": 1,
                    "isRequired": True
                }
            ]
        })

    return pages



In [ ]:
import json, math, os

BATCH_SIZE = 100
OUTPUT_DIR = "/content/survey_batches"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def write_batches(domain, survey_init, survey_final, titles, texts, batch_size=BATCH_SIZE, suffix=""):
    """
    Build SurveyJS batch JSON files with optional suffix in filename.
    Example:
      suffix="test"  -> electronics_test_batch_1.json
      suffix=""      -> electronics_batch_1.json
    """
    print(f"\n🧩 Building pages for {domain} ...")
    pages = build_pages_for_domain(domain, survey_init, survey_final, titles, texts)
    total = len(pages)
    n_batches = math.ceil(total / batch_size)
    print(f"   Total {total} pairs → {n_batches} batches of {batch_size}")

    # construct suffix text
    suffix_part = f"_{suffix}" if suffix else ""

    for b in range(n_batches):
        start = b * batch_size
        end   = min(start + batch_size, total)
        part_pages = pages[start:end]
        survey = {
            "title": f"{domain.capitalize()} Bundle Comparison — Batch {b+1}{suffix_part}",
            "showProgressBar": "top",
            "showQuestionNumbers": "off",
            "pages": part_pages
        }

        out_path = os.path.join(OUTPUT_DIR, f"{domain}{suffix_part}_batch_{b+1}.json")
        with open(out_path, "w") as f:
            json.dump(survey, f, indent=2)

        print(f"   ✓ Wrote {out_path} ({len(part_pages)} pages)")

    print(f"✅ Done: {domain} → {n_batches} batch files in {OUTPUT_DIR}")

# === usage example ===
write_batches(
    "electronics",
    survey_initial_electronic_bundle_indices,
    survey_final_electronic_bundle_indices,
    electronic_metadata["titles"],
    text_electronics,
    suffix="cunt"
)


In [ ]:
# ===== CONFIG =====
import re, json, math, os, random, pandas as pd

CSV_PATHS = {
    "electronics": "/content/LLM4BEAR/3_Human Evaluation/electronic_files_sorted_thumbs.csv",
    "clothing":    "/content/LLM4BEAR/3_Human Evaluation/clothing_files_sorted_thumbs.csv",
    "food":        "/content/LLM4BEAR/3_Human Evaluation/food_files_sorted_thumbs.csv",
}

BATCH_SIZE = 20
OUTPUT_DIR = '/content/drive/My Drive/bundles/surveys/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SURVEY_MAX_WIDTH = 1100        # tweak 1000–1200 to taste
COLUMN_MIN_WIDTH = 460         # tweak 420–500 to taste
GAP_PX = 64

# ===== LOOKUPS =====
def num_from_name(name: str) -> int:
    m = re.search(r"(\d+)", str(name))
    return int(m.group(1)) if m else None

def build_num2url(domain: str) -> dict:
    df = pd.read_csv(CSV_PATHS[domain])
    # Use Drive thumbnail endpoint (renders inside SurveyJS)
    num2uc = {num_from_name(r["name"]): r["imageLink"] for _, r in df.iterrows()}
    def to_thumb(url: str, width=1200):
        return re.sub(r'uc\?export=view&id=([^"&]+)',
                      fr'thumbnail?sz=w{width}&id=\1', url)
    return {k: to_thumb(v) for k, v in num2uc.items()}

maps = {d: build_num2url(d) for d in ["electronics", "clothing", "food"]}

def indices_to_urls_zero_based(domain, idx_list, num2url):
    # +1 because filenames are 1-based
    return [num2url[i + 1] for i in idx_list]

# ===== RENDERING =====
def make_item_block(url, title, desc):
    return f"""
    <div style="border-radius:8px;padding:12px;background:#fff;box-shadow:0 0 0 1px rgba(0,0,0,.06);margin-bottom:16px;">
      <img src="{url}" alt="" style="width:100%;max-height:260px;object-fit:contain;display:block;margin:0 auto 8px;">
      <div style="font-weight:700;margin:6px 0 4px;line-height:1.25;">{title}</div>
      <div style="font-size:14px;line-height:1.35;color:#444;">{desc}</div>
    </div>
    """

def bundle_column_html(domain, idx_vec, titles, texts):
    urls = indices_to_urls_zero_based(domain, idx_vec, maps[domain])
    return "".join(make_item_block(urls[j], titles[i], texts[i]) for j, i in enumerate(idx_vec))

def build_pages_for_domain(domain, survey_init, survey_final, titles, texts):
    pages = []
    for k in range(len(survey_init)):
        # 50/50 left-right flip
        randomised = random.random() < 0.5
        if randomised:
            left_idx, right_idx = survey_final[k], survey_init[k]
            left_label, right_label = "Bundle 1", "Bundle 2"
            rnd_flag = "yes"
        else:
            left_idx, right_idx = survey_init[k], survey_final[k]
            left_label, right_label = "Bundle 1", "Bundle 2"
            rnd_flag = "no"

        left_html  = bundle_column_html(domain, left_idx,  titles, texts)
        right_html = bundle_column_html(domain, right_idx, titles, texts)

        html = f"""
        <!-- randomise: {rnd_flag} -->
        <div style="
          max-width:{SURVEY_MAX_WIDTH}px;margin:0 auto;
          display:flex;gap:{GAP_PX}px;align-items:flex-start;justify-content:center;flex-wrap:nowrap;">
          <div style="flex:1;min-width:{COLUMN_MIN_WIDTH}px;">
            <h4 style="text-align:center;margin:0 0 8px;">{left_label}</h4>
            {left_html}
          </div>
          <div style="flex:1;min-width:{COLUMN_MIN_WIDTH}px;">
            <h4 style="text-align:center;margin:0 0 8px;">{right_label}</h4>
            {right_html}
          </div>
        </div>
        """

        pages.append({
            "name": f"{domain}_pair_{k}",
            "elements": [
                {"type": "html", "name": f"{domain}_pair_{k}_html", "html": html},
                {
                    "type": "radiogroup",
                    "name": f"{domain}_pair_{k}_choice",
                    "title": "Which bundle is better — bundle 1 or bundle 2?",
                    "choices": [
                        {"value": "bundle_1", "text": "Bundle 1 is better"},
                        {"value": "bundle_2", "text": "Bundle 2 is better"}
                    ],
                    "isRequired": True
                },
                {
                    "type": "rating",
                    "name": f"{domain}_pair_{k}_bundle1_rating",
                    "title": "Rate bundle 1:",
                    "rateMin": 1, "rateMax": 5, "rateStep": 1,
                    "isRequired": True
                },
                {
                    "type": "rating",
                    "name": f"{domain}_pair_{k}_bundle2_rating",
                    "title": "Rate bundle 2",
                    "rateMin": 1, "rateMax": 5, "rateStep": 1,
                    "isRequired": True
                }
            ]
        })
    return pages

# ===== BATCH WRITER =====
def write_batches(domain, survey_init, survey_final, titles, texts, batch_size=BATCH_SIZE, suffix=""):
    pages = build_pages_for_domain(domain, survey_init, survey_final, titles, texts)
    total = len(pages)
    n_batches = math.ceil(total / batch_size)
    suffix_part = f"_{suffix}" if suffix else ""
    for b in range(n_batches):
        start, end = b * batch_size, min((b + 1) * batch_size, total)
        part_pages = pages[start:end]
        survey = {
            "title": f"{domain.capitalize()} Bundle Comparison — Batch {b+1}{suffix_part}",
            "showProgressBar": "top",
            "showQuestionNumbers": "off",
            "widthMode": "responsive",      # ✅ official way to control layout width
            "pages": part_pages
        }
        out_path = os.path.join(OUTPUT_DIR, f"{domain}{suffix_part}_batch_{b+1}.json")
        with open(out_path, "w") as f: json.dump(survey, f, indent=2)
        print(f"✓ Wrote {out_path} ({len(part_pages)} pages)")
    print(f"Done: {domain} → {n_batches} batch files -> {OUTPUT_DIR}")

# ===== EXAMPLE CALLS (uncomment and run what you need) =====
write_batches("electronics",
              survey_initial_electronic_bundle_indices,
              survey_final_electronic_bundle_indices,
              electronic_metadata["titles"], text_electronics,
              suffix="")
write_batches("clothing",
              survey_initial_clothing_bundle_indices,
              survey_final_clothing_bundle_indices,
              clothing_metadata["titles"], text_clothing,
              suffix="")
write_batches("food",
              survey_initial_food_bundle_indices,
              survey_final_food_bundle_indices,
              food_metadata["titles"], text_food,
              suffix="")


# All Surveys have been provided in the Github under the folder:
## LLM4BEAR/3_Human Evaluation/surveys/